In [1]:
import os, gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient
import huggingface_hub

# ── Auth ──────────────────────────────────────────────────────
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ.pop("HF_TOKEN", None)
os.environ.pop("HUGGING_FACE_HUB_TOKEN", None)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)

# ── Config ────────────────────────────────────────────────────
REPO = "Arezki-Cherfouh/llama-3.1-8b-med-expert"
SYS  = "You are a clinical AI assistant. Answer clearly and concisely. Reason step-by-step. Do not repeat yourself. If urgent, tell the user to seek emergency care."

# ── Load ──────────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(REPO, token=HF_TOKEN)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    REPO, token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    max_memory={0: "13GiB", 1: "13GiB"},
)
model.eval()
device = next(model.parameters()).device
print(f"✅ Ready on {device}\n")

# ── Chat ──────────────────────────────────────────────────────
def chat(user_input, history=None):
    if history is None:
        history = []
    history.append({"role": "user", "content": user_input})

    prompt = f"System: {SYS}\n\n"
    for turn in history:
        role = "Patient" if turn["role"] == "user" else "Doctor"
        prompt += f"{role}: {turn['content']}\n"
    prompt += "Doctor:"

    inputs = tok(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            pad_token_id=128256,
            eos_token_id=128009,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    reply = tok.decode(new_tokens, skip_special_tokens=True).strip()
    history.append({"role": "assistant", "content": reply})
    print(f"Assistant: {reply}\n")
    return reply, history

# ── Test ──────────────────────────────────────────────────────
history = None
reply, history = chat("What is the first-line treatment for hypertension?", history)
reply, history = chat("What about in pregnant patients?", history)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Ready on cuda:0

Assistant: The initial drug of choice in treating high blood pressure would be thiazide diuretics if there were no contraindications. These drugs act by inhibiting sodium reabsorption at some level along both sides of the distal tubule where most sodium chloride transport occurs.
```2024 Step 1
```

The answer is Thiazides.

Assistant: In pregnancy it’s better to use methyldopa as an alternative because they’re safer for the baby than other medications that could cause birth defects or harm to the fetus while still effectively lowering maternal BP into normal ranges with minimal side effects when used properly.
```2023 Step 2
```



In [2]:
# import os
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# from kaggle_secrets import UserSecretsClient
# import huggingface_hub

# HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
# os.environ.pop("HF_TOKEN", None)
# os.environ.pop("HUGGING_FACE_HUB_TOKEN", None)
# os.environ["HF_TOKEN"] = HF_TOKEN
# os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
# huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)

# REPO = "Arezki-Cherfouh/llama-3.1-8b-med-expert"

# tok = AutoTokenizer.from_pretrained(REPO, token=HF_TOKEN)

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
# )

# model = AutoModelForCausalLM.from_pretrained(
#     REPO, token=HF_TOKEN,
#     quantization_config=bnb_config,
#     device_map="auto",
#     low_cpu_mem_usage=True,
#     max_memory={0: "13GiB", 1: "13GiB"},
# )
# model.eval()
# device = next(model.parameters()).device
# print(f"✅ Ready on {device}\n")

# # ── Diagnostic ───────────────────────────────────────────────
# print("Model type:", type(model))
# print("Config model_type:", model.config.model_type)
# print("\nFirst 5 params:")
# for name, param in list(model.named_parameters())[:5]:
#     print(f"  {name}: {param.shape} {param.dtype}")

# # ── Raw completion test (no chat template) ───────────────────
# prompt = "The first-line treatment for hypertension is"
# inputs = tok(prompt, return_tensors="pt").to(device)
# print(f"\nInput: '{prompt}'")
# print(f"Input token count: {inputs['input_ids'].shape[-1]}")

# with torch.no_grad():
#     out = model.generate(
#         input_ids=inputs["input_ids"],
#         attention_mask=inputs["attention_mask"],
#         max_new_tokens=50,
#         do_sample=False,
#         pad_token_id=128256,
#         eos_token_id=128009,
#     )

# print(f"Output shape: {out.shape}")
# print(f"New token ids: {out[0][inputs['input_ids'].shape[-1]:]}")
# print(f"\nDecoded: {tok.decode(out[0], skip_special_tokens=True)}")